In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(base_url='https://openrouter.ai/api/v1',api_key=os.environ['OPENAI_API_KEY'],model="gpt-3.5-turbo",temperature=0)

## **RAG IMPLEMENTATION WITH PDF File** ##

#### **STEP 1: Extracting text from PDFs** ####

In [3]:
from langchain_community.document_loaders import PyPDFLoader 

In [4]:
pdf_path='./Docs/fabric-onelake.pdf'
loader=PyPDFLoader(pdf_path)
docs=loader.load()

### ***Creating own Metadata*** ###

In [5]:
for i in docs:
    i.metadata={'producer': 'Microsoft Learn PDF 1.0.26270.03', 'creator': 'Microsoft Learn', 'title': 'fabric onelake | Microsoft Learn', 'source': './Docs/fabric-onelake.pdf'}

#### **STEP 2: Chunking the Documents** ####

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks=splitter.split_documents(docs)

#### **STEP 3: Creating Embeddings for the CHUNKS** ####

In [7]:
from langchain_openai import OpenAIEmbeddings

In [8]:
embedding_model=OpenAIEmbeddings(openai_api_base='https://openrouter.ai/api/v1',model="openai/text-embedding-3-small")

#### **STEP4: Create and Store Embeddings of the chunks in Local Vector DB** ####

In [9]:
from langchain_community.vectorstores import Chroma

In [10]:
vectorstore=Chroma.from_documents(documents=chunks,embedding=embedding_model,persist_directory='./VectorStore/')

In [11]:
context=vectorstore.similarity_search("Why do we need onelake?",k=5)

### ***Reuse the stored data*** ###

In [12]:
vectorestore_persist = Chroma(persist_directory="./VectorStore/",embedding_function=embedding_model)

In [13]:
vectorestore_persist.similarity_search("Why do we need onelake?")

[Document(metadata={'creator': 'Microsoft Learn', 'title': 'fabric onelake | Microsoft Learn', 'producer': 'Microsoft Learn PDF 1.0.26270.03', 'source': './Docs/fabric-onelake.pdf'}, page_content="diagnostics\nBefore OneLake, organizations often created multiple lakes for different business groups, which\nled to extra overhead for managing multiple resources. This siloed approach made it difficult to\ncollaborate across teams, slowed down data projects, and increased the risk of duplication.\nOneLake solves these challenges by giving you a central data access point for the entire\norganization. Every Fabric tenant comes with a single OneLake instance. You can't delete OneLake"),
 Document(metadata={'source': './Docs/fabric-onelake.pdf', 'producer': 'Microsoft Learn PDF 1.0.26270.03', 'creator': 'Microsoft Learn', 'title': 'fabric onelake | Microsoft Learn'}, page_content="What is OneLake?\nMicrosoft OneLake is a unified data lake for your whole organization. Every Microsoft Fabric\nten

In [12]:
llm.invoke("Tell me about microsoft fablic - onelake - use this context:{context}")

AIMessage(content='Microsoft Fablic - OneLake is a cloud-based platform that offers a wide range of services and tools for data management, analytics, and visualization. It allows users to easily store, analyze, and share data across different devices and platforms. With its user-friendly interface and powerful features, Microsoft Fablic - OneLake is a popular choice for businesses and organizations looking to harness the power of big data for decision-making and strategic planning.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 24, 'total_tokens': 109, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 0.0001395, 'is_byok': Fal